# — Data Quality Plan —— (DQP)

## Core Objectives
1. Read the raw training dataset `ppr-group-25208508-train.csv`.
2. Apply rule-based cleaning (types, missing values, duplicates, outliers, and business rules).
3. Produce:
   - `ppr-group-25208508-clean-final.csv` (cleaned output in this notebook)
   - a step-by-step processing log for auditability.


## 1) Imports & Settings
Import required libraries and configure pandas display options for readable outputs.


In [1]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

## 2) Load Dataset
Load the training dataset `ppr-group-25208508-train.csv`.


In [2]:
from pathlib import Path

DATA_PRIMARY = Path("ppr-group-25208508-train.csv")

if DATA_PRIMARY.exists():
    data_path = DATA_PRIMARY
else:
    raise FileNotFoundError("Cannot find ppr-group-25208508-train.csv (or fallback preview).")

df_raw = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df_raw.shape)
df_raw.head()

FileNotFoundError: Cannot find ppr-group-25208508-train.csv (or fallback preview).

## 3) Quick Structure Check (Before Cleaning)
Report column names, missingness, dtypes, and duplicate-row count as the pre-cleaning baseline.


In [ ]:
def quick_profile(df: pd.DataFrame, title: str):
    print("="*80)
    print(title)
    print("Shape:", df.shape)
    print("\nDtypes:")
    display(df.dtypes.to_frame("dtype"))
    print("\nMissing (top 15):")
    miss = df.isna().sum().sort_values(ascending=False)
    display(miss.head(15).to_frame("missing_count"))
    print("\nDuplicate rows:", int(df.duplicated().sum()))
    print("="*80)

quick_profile(df_raw, "BEFORE CLEANING")

BEFORE CLEANING
Shape: (54000, 9)

Dtypes:


,dtype
Date of Sale (dd/mm/yyyy),str
Address,str
County,str
Eircode,str
Price (€),str
Not Full Market Price,str
VAT Exclusive,str
Description of Property,str
Property Size Description,str



Missing (top 15):


,missing_count
Property Size Description,51164
Eircode,37108
Date of Sale (dd/mm/yyyy),0
Address,0
County,0
Price (€),0
Not Full Market Price,0
VAT Exclusive,0
Description of Property,0



Duplicate rows: 9


## 4) Utility Functions
This section defines reusable helpers for text normalization, price parsing, address feature extraction, and process logging.

Function summary:
- `DQLog`: records per-step row/column changes and notes for an auditable cleaning trail.
- `clean_whitespace(x)`: trims text and normalizes repeated spaces.
- `parse_price_to_numeric(series)`: converts price strings (e.g., `€123,456.78`) to numeric values.
- `extract_dublin_district(addr)`: extracts Dublin district patterns (e.g., `Dublin 4`, `D6W`).
- `extract_area_token(addr)`: derives a coarse location token from address text without external APIs.

These helpers improve consistency, explainability, and reproducibility in downstream steps.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class DQLog:
    steps: list = field(default_factory=list)

    def add(self, step: str, before_shape, after_shape, notes: str = ""):
        br, bc = before_shape
        ar, ac = after_shape
        self.steps.append({
            "step": step,
            "rows_before": br, "rows_after": ar, "rows_delta": ar - br,
            "cols_before": bc, "cols_after": ac, "cols_delta": ac - bc,
            "notes": notes
        })

    def to_frame(self):
        return pd.DataFrame(self.steps)

log = DQLog()

def clean_whitespace(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def parse_price_to_numeric(series: pd.Series) -> pd.Series:
    # Handles values like "€123,456.78" or "123,456"
    s = series.astype(str).str.strip()
    s = s.str.replace("€", "", regex=False).str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def extract_dublin_district(addr: str):
    """Extract Dublin district if pattern appears.
    Examples: 'Dublin 4', 'Dublin 6W', 'D6', 'D6W' -> 'Dublin 4' / 'Dublin 6W'
    Returns NaN if not found.
    """
    if pd.isna(addr):
        return np.nan
    s = str(addr).lower()
    m = re.search(r"\bdublin\s*(\d{1,2}w?)\b", s)
    if m:
        return "Dublin " + m.group(1).upper()
    m = re.search(r"\bd(\d{1,2}w?)\b", s)  # e.g., D6W / D4
    if m:
        return "Dublin " + m.group(1).upper()
    return np.nan

def extract_area_token(addr: str):
    """Heuristic area token from Address without external APIs.
    - Split by commas
    - Take the 2nd last token if available (often locality/suburb), else last
    - Normalize to lowercase alphanum + space + hyphen
    """
    if pd.isna(addr):
        return np.nan
    parts = [p.strip() for p in str(addr).split(",") if p.strip()]
    if len(parts) == 0:
        return np.nan
    cand = parts[-2] if len(parts) >= 2 else parts[-1]
    cand = cand.lower()
    cand = re.sub(r"[^a-z0-9\s\-]", "", cand)
    cand = re.sub(r"\s+", " ", cand).strip()
    return cand if cand else np.nan

## 5) Start Cleaning — Work on a Copy
Use a working copy (`df`) instead of modifying `df_raw` directly.


In [ ]:
df = df_raw.copy()

## 6) Basic Standardization
- Normalize whitespace in text fields.
- Parse date fields.
- Convert `Price (€)` to numeric.
- Create time-derived features (`Sale Year`, `Sale Month`) to align with DQR.


In [ ]:
before = df.shape

# Text columns: trim and normalize spaces
text_cols = [c for c in df.columns if df[c].dtype == "object"]
for c in text_cols:
    df[c] = df[c].apply(clean_whitespace)

# Date parsing
date_col = "Date of Sale (dd/mm/yyyy)"
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")

# Price parsing
price_col = "Price (€)"
if price_col in df.columns:
    df[price_col] = parse_price_to_numeric(df[price_col])

# Simple time features (match DQR naming)
if date_col in df.columns:
    df["Sale Year"] = df[date_col].dt.year
    df["Sale Month"] = df[date_col].dt.month

after = df.shape
log.add("Standardize text + parse date + parse price + create Sale Year/Month", before, after)
df.head()

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month
0,2016-09-30,"28 BRACKEN COURT, DONNYBROOK, CORK",Cork,NaN,181000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,9
1,2016-12-20,"2 AN CLOCHAR, CONVENT RD, DONERAILE",Cork,NaN,50152.49,No,Yes,New Dwelling house /Apartment,less than 38 sq metres,2016,12
2,2016-09-28,"Apartment 7 The Court, Clonattin, Gorey",Wexford,NaN,62171.81,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9
3,2016-09-16,"6 Monalin, Wicklow Hills, Newtownmountkennedy",Wicklow,NaN,223348.00,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9
4,2016-01-29,"18 Lislea, Frascati Park, Blackrock",Dublin,NaN,310000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,1


## 7) Drop Fully Empty / Constant Columns (If Any)
Detect and remove fully empty columns or constant columns, then log the decision.


In [ ]:
before = df.shape

# Fully empty columns
empty_cols = [c for c in df.columns if df[c].isna().all()]
# Constant columns (single unique non-null value)
const_cols = []
for c in df.columns:
    nun = df[c].nunique(dropna=True)
    if nun == 1 and not df[c].isna().all():
        const_cols.append(c)

drop_cols = sorted(set(empty_cols + const_cols))
if drop_cols:
    df = df.drop(columns=drop_cols)

after = df.shape
notes = f"empty_cols={empty_cols}; const_cols={const_cols}"
log.add("Drop fully-empty / constant columns", before, after, notes)
drop_cols

[]

## 8) Duplicate Rows
Remove exact duplicate rows and record the number removed (currently 9 rows).


In [ ]:
before = df.shape
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()
after = df.shape
log.add("Drop duplicate rows", before, after, f"duplicates_removed={dup_count}")
dup_count

9

## 9) Core Field Validations
Apply minimum validity checks:
- `Price (€)` must be positive.
- Sale date must be valid/parseable.
- `County` must be non-empty.


In [ ]:
before = df.shape

reject_mask = pd.Series(False, index=df.index)

# price must be > 0
if price_col in df.columns:
    reject_mask |= df[price_col].isna() | (df[price_col] <= 0)

# date must be valid
if date_col in df.columns:
    reject_mask |= df[date_col].isna()

# county not null (optional but usually reasonable)
if "County" in df.columns:
    reject_mask |= df["County"].isna() | (df["County"].astype(str).str.strip() == "")

reject_count = int(reject_mask.sum())
df_rejected = df.loc[reject_mask].copy()
df = df.loc[~reject_mask].copy()

after = df.shape
log.add("Filter invalid core rows (price/date/county)", before, after, f"rejected_rows={reject_count}")

print("Rejected rows:", reject_count)
df_rejected.head()

Rejected rows: 0


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month


## 10) Address Handling — Feature Engineering then Drop Raw Address

### Background and Problem
`Address` is a high-cardinality free-text field with inconsistent formatting and high sparsity when encoded directly. DQR also identified high missingness in `Eircode`, so raw location fields are not reliable as-is.

### Strategy (Extract First, Drop Later)
1. Engineer compact, explainable location/morphology features from `Address`.
2. Drop raw `Address` after feature extraction to reduce noise, dimensionality, and privacy risk.

### Why Function-Based Feature Construction
- **Consistency**: standardizes text and parsing logic.
- **Explainability**: engineered features are interpretable.
- **Reproducibility**: deterministic rules, minimal external dependency.
- **Auditability**: each transformation is traceable in the processing log.

### Relation to `Address_to_GPS` Pipeline
The `Address_to_GPS/` workflow (geocoding -> district extraction -> rematch -> map validation) was used as supporting exploration to validate geographic signal extraction. For the main DQP pipeline, we keep an API-light, reproducible approach while preserving the same feature-engineering intent.

### Advantages
- Reduces dimensionality and overfitting risk.
- Improves generalization by reducing raw-text noise.
- Preserves useful geographic signal via engineered features.
- Keeps the pipeline reproducible and report-ready.

### Final Decision
Use `Address` only as an intermediate source for feature generation, then remove the raw column.


In [ ]:
before = df.shape

addr_col = "Address"
if addr_col in df.columns:
    # Basic normalization already done; extract features
    df["is_apartment"] = df[addr_col].str.contains(r"\b(apartment|apt|unit|flat)\b", case=False, na=False).astype(int)
    df["has_number"] = df[addr_col].str.contains(r"\b\d+\b", na=False).astype(int)

    df["dublin_district"] = df[addr_col].apply(extract_dublin_district)
    df["dublin_district_missing"] = df["dublin_district"].isna().astype(int)

    df["area_token"] = df[addr_col].apply(extract_area_token)
    df["area_token_missing"] = df["area_token"].isna().astype(int)

    # Finally drop raw Address
    df = df.drop(columns=[addr_col])

after = df.shape
log.add("Scheme A: derive address features then drop Address", before, after, "Added: is_apartment, has_number, dublin_district, area_token + missing flags")
df.head()

/var/folders/jw/0ttqd5gn6_zf0xv62ypjvn9r0000gn/T/ipykernel_76065/3198737329.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_apartment"] = df[addr_col].str.contains(r"\b(apartment|apt|unit|flat)\b", case=False, na=False).astype(int)


,Date of Sale (dd/mm/yyyy),County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month,is_apartment,has_number,dublin_district,dublin_district_missing,area_token,area_token_missing
0,2016-09-30,Cork,NaN,181000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,9,0,1,NaN,1,donnybrook,0
1,2016-12-20,Cork,NaN,50152.49,No,Yes,New Dwelling house /Apartment,less than 38 sq metres,2016,12,0,1,NaN,1,convent rd,0
2,2016-09-28,Wexford,NaN,62171.81,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9,1,1,NaN,1,clonattin,0
3,2016-09-16,Wicklow,NaN,223348.00,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9,0,1,NaN,1,wicklow hills,0
4,2016-01-29,Dublin,NaN,310000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,1,0,1,NaN,1,frascati park,0


### 10.1) Address-Derived Feature Distribution and `area_token` Cardinality Reduction (N >= 50)

This step has two goals:
1. Summarize distributions of address-derived features (`is_apartment`, `has_number`, `dublin_district`, `area_token`) to verify reasonableness and interpretability.
2. Reduce high cardinality in `area_token` using frequency thresholding:
   - Keep tokens with frequency **>= 50**.
   - Merge lower-frequency tokens into **`Other`**.

This retains major location signal while controlling feature explosion and improving model stability.


In [ ]:
# --- Address-derived feature distribution ---
addr_feats = [
    "is_apartment", "has_number",
    "dublin_district", "dublin_district_missing",
    "area_token", "area_token_missing"
]
addr_feats = [c for c in addr_feats if c in df.columns]

summary = pd.DataFrame({
    "dtype": df[addr_feats].dtypes.astype(str),
    "missing_rate_%": (df[addr_feats].isna().mean() * 100).round(2),
    "n_unique": df[addr_feats].nunique(dropna=True),
}).sort_values("n_unique", ascending=False)

display(summary)

# Binary feature counts (0/1)
binary_cols = [c for c in ["is_apartment", "has_number", "dublin_district_missing"] if c in df.columns]
for c in binary_cols:
    vc = df[c].value_counts(dropna=False)
    pct = (vc / len(df) * 100).round(2)
    print(f"\n{c} counts:")
    display(pd.DataFrame({"count": vc, "pct_%": pct}))

# dublin_district distribution (top 20)
if "dublin_district" in df.columns:
    print("\ndublin_district (top 20):")
    display(df["dublin_district"].value_counts(dropna=False).head(20).to_frame("count"))

# --- area_token cardinality reduction (frequency thresholding) ---
MIN_COUNT = 50

if "area_token" in df.columns:
    before = df.shape

    vc = df["area_token"].value_counts(dropna=False)
    n_unique = int(vc.shape[0])
    n_once = int((vc == 1).sum())

    keep = vc[vc >= MIN_COUNT].index
    kept_cardinality = int(len(keep))
    coverage_rows = int(vc[vc >= MIN_COUNT].sum())
    coverage_pct = round(coverage_rows / len(df) * 100, 2)

    df["area_token_reduced"] = df["area_token"].where(df["area_token"].isin(keep), "Other")

    # Stats after reduction
    vc_reduced = df["area_token_reduced"].value_counts(dropna=False)
    reduced_cardinality = int(vc_reduced.shape[0])
    other_rows = int((df["area_token_reduced"] == "Other").sum())
    other_pct = round(other_rows / len(df) * 100, 2)

    after = df.shape
    notes = (
        f"MIN_COUNT={MIN_COUNT}, unique_before={n_unique}, once_before={n_once} "
        f"({round(n_once/max(n_unique,1)*100,2)}% of unique), kept_cardinality={kept_cardinality}, "
        f"coverage={coverage_pct}%, reduced_cardinality={reduced_cardinality}, Other={other_pct}%"
    )
    log.add("Reduce area_token cardinality (freq>=50 -> keep else Other)", before, after, notes)

    print("\narea_token reduction summary:")
    print(notes)
    print("\narea_token_reduced (top 20):")
    display(vc_reduced.head(20).to_frame("count"))
else:
    print("area_token column not found; skipped reduction.")


,dtype,missing_rate_%,n_unique
area_token,str,0.00,14683
dublin_district,str,83.43,24
is_apartment,int64,0.00,2
has_number,int64,0.00,2
dublin_district_missing,int64,0.00,2
area_token_missing,int64,0.00,1



is_apartment counts:


,count,pct_%
is_apartment,,
0,50901,94.28
1,3090,5.72



has_number counts:


,count,pct_%
has_number,,
1,42347,78.43
0,11644,21.57


...

,count,pct_%
dublin_district_missing,,
1,45046,83.43
0,8945,16.57



dublin_district (top 20):


,count
dublin_district,
NaN,45046
Dublin 15,1035
Dublin 8,631
Dublin 4,540
Dublin 7,538
Dublin 9,536
Dublin 12,520
Dublin 24,485
Dublin 11,459



area_token reduction summary:
MIN_COUNT=50, unique_before=14683, once_before=9602 (65.4% of unique), kept_cardinality=133, coverage=21.24%, reduced_cardinality=134, Other=78.76%

area_token_reduced (top 20):


,count
area_token_reduced,
Other,42525
blackrock,224
clondalkin,218
lucan,214
swords,192
castleknock,173
dundalk,167
balbriggan,160
letterkenny,158


## 11) VAT Rule
For rows where `Description of Property == 'New Dwelling house /Apartment'` and `VAT Exclusive == Yes`, adjust price using VAT (13.5%).

Feature handling:
- Keep original `Price (€)`.
- Create `Price_Adjusted_VAT`.
- Create `vat_adjusted_flag` to mark adjusted rows.


In [ ]:
before = df.shape

VAT_RATE = 0.135
df["vat_adjusted_flag"] = 0

if price_col in df.columns and "Description of Property" in df.columns and "VAT Exclusive" in df.columns:
    cond = (
        (df["Description of Property"].astype(str).str.strip() == "New Dwelling house /Apartment") &
        (df["VAT Exclusive"].astype(str).str.strip().str.lower() == "yes") &
        (df[price_col].notna())
    )
    df["Price_Adjusted_VAT"] = df[price_col].copy()
    df.loc[cond, "Price_Adjusted_VAT"] = df.loc[cond, price_col] * (1 + VAT_RATE)
    df.loc[cond, "vat_adjusted_flag"] = 1
    affected = int(cond.sum())
    note = f"VAT_RATE={VAT_RATE}, affected_rows={affected}"
else:
    df["Price_Adjusted_VAT"] = df.get(price_col, np.nan)
    note = "Required columns missing; created Price_Adjusted_VAT as copy/NaN."

after = df.shape
log.add("VAT adjustment: create Price_Adjusted_VAT + vat_adjusted_flag", before, after, note)
note

'VAT_RATE=0.135, affected_rows=9513'

## 12) Clamping after VAT Adjustment
`Price_Adjusted_VAT_Clamped` is used as the standardized target for downstream analysis/modeling because it addresses two issues:
- inconsistent price definition (VAT-inclusive vs VAT-exclusive),
- heavy-tailed distribution and extreme outliers.

Applied rule:
- compute the 5th and 95th percentiles on `Price_Adjusted_VAT`,
- clip values outside this range,
- keep original/intermediate columns for auditability.

In DQR, multiple clamping levels can still be compared for exploratory analysis, but this notebook sets the final DQP cleaning rule.


In [ ]:
before = df.shape

if "Price_Adjusted_VAT" in df.columns:
    p05_vat = df["Price_Adjusted_VAT"].quantile(0.05)
    p95_vat = df["Price_Adjusted_VAT"].quantile(0.95)

    df["Price_Adjusted_VAT_Clamped"] = df["Price_Adjusted_VAT"].clip(lower=p05_vat, upper=p95_vat)

    changed = (df["Price_Adjusted_VAT_Clamped"] != df["Price_Adjusted_VAT"]).sum()
    notes = f"p05_vat={p05_vat:.2f}, p95_vat={p95_vat:.2f}, clamped_rows={int(changed)}"
else:
    notes = "Price_Adjusted_VAT not found; skipped VAT clamping."

after = df.shape
log.add("VAT price clamping: create Price_Adjusted_VAT_Clamped using p05–p95", before, after, notes)

notes

'p05_vat=60000.00, p95_vat=721137.50, clamped_rows=5333'

## 13) High Missingness Columns — Keep vs Drop
Based on DQR findings (e.g., `Property Size Description`, `Eircode`), we apply a conservative missing-data policy:
- If missingness is extremely high and analytical value is low: drop.
- If potentially useful: keep with missing flags and appropriate imputation.

A configurable threshold is used: `DROP_MISSING_PCT = 90.0`.


In [ ]:
before = df.shape

DROP_MISSING_PCT = 90.0

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
candidates = missing_pct[missing_pct >= DROP_MISSING_PCT].index.tolist()

# For transparency, we DO NOT auto-drop everything blindly.
# We drop only columns that are BOTH high-missing and clearly low-utility free-text.
auto_drop_allowlist = set([
    "Property Size Description"
])

to_drop = [c for c in candidates if c in auto_drop_allowlist]

if to_drop:
    df = df.drop(columns=to_drop)

after = df.shape
log.add("High-missing columns: optional drop (threshold-based)", before, after, f"DROP_MISSING_PCT={DROP_MISSING_PCT}, candidates={len(candidates)}, dropped={to_drop}")
missing_pct.head(15).to_frame("missing_%")

,missing_%
Property Size Description,94.749125
dublin_district,83.432424
Eircode,68.715156
Date of Sale (dd/mm/yyyy),0.000000
has_number,0.000000
Price_Adjusted_VAT,0.000000
vat_adjusted_flag,0.000000
area_token_reduced,0.000000
area_token_missing,0.000000
area_token,0.000000


### 13.1) Explicit `Eircode` Drop
`Eircode` is explicitly removed and logged due to high missingness and limited utility as a stable modeling feature in this pipeline.


In [ ]:
# Explicitly drop Eircode (group decision)
before = df.shape
if "Eircode" in df.columns:
    df = df.drop(columns=["Eircode"])
after = df.shape
log.add("Drop Eircode column explicitly (group decision)", before, after)


## 14) Fill Missing Values (Conservative, Aligned with Downstream Modeling)
### DQP handling in this notebook (training data)
To produce a modeling-ready cleaned dataset:
- categorical fields -> fill with `Unknown`
- numeric fields -> fill with median

This matches the cleaning logic used to generate `train_data_clean.csv` in `model_training/linear_regression.ipynb`.

### Connection to downstream test-set handling
In model evaluation (`test_data.csv`), the implemented flow mainly uses rule-based cleaning plus invalid-row filtering (text normalization, parsing, feature construction, then filtering). The test path is therefore not a full symmetric all-field `Unknown/median` imputation pipeline.

### Why this design
- Training set: enforce a stable, reproducible no-NaN dataset for model fitting.
- Test set: keep preprocessing closer to realistic inference conditions with minimal extra assumptions.


In [ ]:
before = df.shape

# Identify numeric vs categorical/object
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

# Fill categorical
for c in cat_cols:
    # Skip date column if still datetime (should not be in cat)
    if c == date_col:
        continue
    df[c] = df[c].fillna("Unknown")

# Fill numeric
for c in num_cols:
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(med)

after = df.shape
log.add("Fill missing values (cat='Unknown', num=median)", before, after)
df.head()

,Date of Sale (dd/mm/yyyy),County,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Sale Year,Sale Month,is_apartment,has_number,dublin_district,dublin_district_missing,area_token,area_token_missing,area_token_reduced,vat_adjusted_flag,Price_Adjusted_VAT,Price_Adjusted_VAT_Clamped
0,2016-09-30,Cork,181000.00,No,No,Second-Hand Dwelling house /Apartment,2016,9,0,1,Unknown,1,donnybrook,0,donnybrook,0,181000.00000,181000.00000
1,2016-12-20,Cork,50152.49,No,Yes,New Dwelling house /Apartment,2016,12,0,1,Unknown,1,convent rd,0,Other,1,56923.07615,60000.00000
2,2016-09-28,Wexford,62171.81,No,Yes,New Dwelling house /Apartment,2016,9,1,1,Unknown,1,clonattin,0,Other,1,70565.00435,70565.00435
3,2016-09-16,Wicklow,223348.00,No,Yes,New Dwelling house /Apartment,2016,9,0,1,Unknown,1,wicklow hills,0,Other,1,253499.98000,253499.98000
4,2016-01-29,Dublin,310000.00,No,No,Second-Hand Dwelling house /Apartment,2016,1,0,1,Unknown,1,frascati park,0,Other,0,310000.00000,310000.00000


## 15) Post-Clean Validation
Checks include:
- whether missing values remain,
- whether the price column is strictly positive,
- whether key assumptions hold after cleaning.


In [ ]:
quick_profile(df, "AFTER CLEANING (POST-VALIDATION)")

# Core checks
assert (df[price_col] > 0).all(), "Found non-positive prices after cleaning."
if date_col in df.columns:
    assert df[date_col].notna().all(), "Found invalid dates after cleaning."

# Show remaining missing
remaining_missing = df.isna().sum().sum()
print("Total remaining missing values:", int(remaining_missing))

AFTER CLEANING (POST-VALIDATION)
Shape: (53991, 18)

Dtypes:


,dtype
Date of Sale (dd/mm/yyyy),datetime64[us]
County,str
Price (€),float64
Not Full Market Price,str
VAT Exclusive,str
Description of Property,str
Sale Year,int32
Sale Month,int32
is_apartment,int64
has_number,int64



Missing (top 15):


,missing_count
Date of Sale (dd/mm/yyyy),0
County,0
Price_Adjusted_VAT,0
vat_adjusted_flag,0
area_token_reduced,0
area_token_missing,0
area_token,0
dublin_district_missing,0
dublin_district,0
has_number,0



Duplicate rows: 668
Total remaining missing values: 0


## 16) DQ Processing Log
Summarize row/column impact for each cleaning step. The log serves as an auditable evidence trail.
A more complete DQLog usage is also implemented in `model_training/linear_regression.ipynb`.


In [ ]:
log_df = log.to_frame()
display(log_df)

print("Final shape:", df.shape)

,step,rows_before,rows_after,rows_delta,cols_before,cols_after,cols_delta,notes
0,Standardize text + parse date + parse price + ...,54000,54000,0,9,11,2,
1,Drop fully-empty / constant columns,54000,54000,0,11,11,0,empty_cols=[]; const_cols=[]
2,Drop duplicate rows,54000,53991,-9,11,11,0,duplicates_removed=9
3,Filter invalid core rows (price/date/county),53991,53991,0,11,11,0,rejected_rows=0
4,Scheme A: derive address features then drop Ad...,53991,53991,0,11,16,5,"Added: is_apartment, has_number, dublin_distri..."
5,Reduce area_token cardinality (freq>=50 -> kee...,53991,53991,0,16,17,1,"MIN_COUNT=50, unique_before=14683, once_before..."
6,VAT adjustment: create Price_Adjusted_VAT + va...,53991,53991,0,17,19,2,"VAT_RATE=0.135, affected_rows=9513"
7,VAT price clamping: create Price_Adjusted_VAT_...,53991,53991,0,19,20,1,"p05_vat=60000.00, p95_vat=721137.50, clamped_r..."
8,High-missing columns: optional drop (threshold...,53991,53991,0,20,19,-1,"DROP_MISSING_PCT=90.0, candidates=1, dropped=[..."
9,Drop Eircode column explicitly (group decision),53991,53991,0,19,18,-1,


Final shape: (53991, 18)


## 17) Export Clean Dataset
Export the final cleaned dataset to `ppr-group-25208508-clean-final.csv`.


In [ ]:
OUT_CSV = "ppr-group-25208508-clean-final.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: ppr-group-25208508-preview.csv


## 18) Quick Peek of Output
Sample a few rows to verify the exported dataset format and values.


In [ ]:
df.sample(5, random_state=42)

,Date of Sale (dd/mm/yyyy),County,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Sale Year,Sale Month,is_apartment,has_number,dublin_district,dublin_district_missing,area_token,area_token_missing,area_token_reduced,vat_adjusted_flag,Price_Adjusted_VAT,Price_Adjusted_VAT_Clamped
45584,2023-09-15,Westmeath,299950.0,No,No,Second-Hand Dwelling house /Apartment,2023,9,0,1,Unknown,1,lakepoint park,0,Other,0,299950.0,299950.0
21937,2019-04-26,Cavan,130000.0,No,No,Second-Hand Dwelling house /Apartment,2019,4,0,1,Unknown,1,swellan,0,Other,0,130000.0,130000.0
1319,2016-12-20,Dublin,209000.0,No,No,Second-Hand Dwelling house /Apartment,2016,12,0,0,Unknown,1,balgriffin,0,Other,0,209000.0,209000.0
4640,2016-07-11,Galway,93000.0,No,No,Second-Hand Dwelling house /Apartment,2016,7,0,0,Unknown,1,doughuiska,0,Other,0,93000.0,93000.0
46394,2023-06-14,Wexford,102500.0,No,No,Second-Hand Dwelling house /Apartment,2023,6,0,1,Unknown,1,faythe,0,Other,0,102500.0,102500.0
